# Cancer Detection from Medical Images using Deep Learning (CNN)
**Tech Stack:** Python, TensorFlow/Keras, OpenCV (`cv2`), NumPy, Matplotlib

This notebook implements an end-to-end medical image classification pipeline:
1. **Image Preprocessing & Normalization**: OpenCV resizing, Gaussian denoising, CLAHE local contrast enhancement, and `[0, 1]` pixel intensity scaling.
2. **Data Augmentation**: Geometric and photometric transformations using OpenCV and `tensorflow.keras.preprocessing.image.ImageDataGenerator`.
3. **CNN Classification**: A 4-block Deep Convolutional Neural Network (`Conv2D`, `BatchNormalization`, `MaxPooling2D`, `Dropout`, `GlobalAveragePooling2D`, `Dense`).
4. **Cancerous vs. Non-Cancerous Prediction**: Test evaluation (Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrix) and single-image diagnostic prediction.

In [ ]:
import os
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt

import config
from generate_sample_dataset import build_sample_dataset
from preprocessing import preprocess_image, load_dataset_split, visualize_preprocessing_stages
from augmentation import create_augmented_dataset, get_keras_image_data_generator, save_augmentation_preview
from model import TF_AVAILABLE, build_cancer_detection_cnn, get_training_callbacks
import train
import predict

print('OpenCV version:', cv2.__version__)
print('NumPy version:', np.__version__)
print('TensorFlow available:', TF_AVAILABLE)

## 1. Prepare / Load Medical Image Dataset
If a custom dataset is not yet placed in `dataset/`, the cell below generates synthetic H&E histopathology tissue patches (`non_cancerous` vs. `cancerous`).

In [ ]:
if not (config.TRAIN_DIR / 'cancerous').exists():
    summary = build_sample_dataset()
    print('Generated sample dataset:', summary)

X_train, y_train, train_files = load_dataset_split(config.TRAIN_DIR)
X_val, y_val, val_files = load_dataset_split(config.VAL_DIR)
X_test, y_test, test_files = load_dataset_split(config.TEST_DIR)
print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

## 2. Image Preprocessing, Normalization & Data Augmentation

In [ ]:
visualize_preprocessing_stages(Path(train_files[0]), config.OUTPUT_DIR / 'preprocessing_pipeline.png')
save_augmentation_preview(X_train[0], config.OUTPUT_DIR / 'data_augmentation_samples.png')
X_train_aug, y_train_aug = create_augmented_dataset(X_train, y_train, multiplier=2)
print('Augmented training set shape:', X_train_aug.shape)

## 3. Train & Evaluate the CNN Classifier

In [ ]:
metrics, test_files, y_test, y_prob = train.run_training_pipeline()
print('Evaluation Metrics:', metrics)

## 4. Predict Cancerous vs. Non-Cancerous on Input Images

In [ ]:
sample_result = predict.predict_single_image(
    Path(test_files[0]),
    config.OUTPUT_DIR / 'single_image_inference.png'
)
print(sample_result)